# Distances Between Observations

---
Title: "Title"
format:
    html:
        embed-resources: true
---

Read this notebook from top to bottom and fill in the code as you go. Work together and discuss with other students in the class. Try to resolve any errors on your own first, but don't get stuck; ask for help!

In addition to writing and running code, be sure to examine any output and interpret the results before moving on.

For many of these questions, there are several approaches, and there is no single right answer. You should try a few different things and compare with your classmates.

We will use `scikit-learn` extensively later, but for this activity you might want to stick with `pandas`.


In [1]:
import pandas as pd
import numpy as np

## Ames - Recommending Similar Homes

1\. Suppose that you really like house 0 in the Ames housing data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar (based on these 3 variables). Do they make sense?

**_Think:_ If the goal is to find a "good deal" on a similar house, should sale price be included as a variable in your distance metric?**

In [2]:
# YOUR CODE HERE. ADD CELLS AS NEEDED

ames = pd.read_csv(
    "https://raw.githubusercontent.com/kevindavisross/data301/main/data/AmesHousing.txt",
    sep="\t"
)

features = ["Gr Liv Area", "Bedroom AbvGr", "Full Bath"]

house_0 = ames.loc[0]
target = house_0[features]

cheaper = ames.loc[ames["SalePrice"] < house_0["SalePrice"]].copy()

feature_sd = ames[features].std()

cheaper["distance"] = (
    ((cheaper[features] - target) / feature_sd)
    .pow(2)
    .sum(axis=1)
    .pow(0.5)
)

recommendations = cheaper.sort_values("distance").head(10)

print("House 0:")
display(ames.loc[[0], features + ["SalePrice"]])

print("Closest cheaper homes:")
display(recommendations[["Order"] + features + ["SalePrice", "distance"]])

House 0:


,Gr Liv Area,Bedroom AbvGr,Full Bath,SalePrice
0,1656,3,1,215000


Closest cheaper homes:


,Order,Gr Liv Area,Bedroom AbvGr,Full Bath,SalePrice,distance
1550,1551,1656,3,1,126000,0.0
1532,1533,1660,3,1,188700,0.007913
1895,1896,1652,3,1,200000,0.007913
1226,1227,1661,3,1,165500,0.009891
1940,1941,1647,3,1,153000,0.017804
1357,1358,1666,3,1,161000,0.019782
291,292,1666,3,1,100000,0.019782
758,759,1666,3,1,135000,0.019782
620,621,1668,3,1,140000,0.023738
1328,1329,1668,3,1,108000,0.023738




```
# This is formatted as code
```

**Yes, the recommendations make sense based on living area, bedrooms, and bathrooms. However, the large price differences suggest that other features, such as neighborhood, condition, and age, should also be considered.
**

2\. Continuing part 1. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms, **and House Style** --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

In [3]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
numeric_vars = ["Gr Liv Area", "Bedroom AbvGr", "Full Bath"]
house_0 = ames.iloc[0]

cheaper = ames[ames["SalePrice"] < house_0["SalePrice"]].copy()

numeric_distance = (
    ((cheaper[numeric_vars] - house_0[numeric_vars]) / ames[numeric_vars].std())
    .pow(2)
    .sum(axis=1)
    .pow(0.5)
)

style_difference = (cheaper["House Style"] != house_0["House Style"]).astype(int)

cheaper["distance"] = (numeric_distance**2 + style_difference**2).pow(0.5)

similar_homes = cheaper.sort_values("distance").head(10)

display(
    similar_homes[
        ["Order", "Gr Liv Area", "Bedroom AbvGr", "Full Bath",
         "House Style", "SalePrice", "distance"]
    ]
)

,Order,Gr Liv Area,Bedroom AbvGr,Full Bath,House Style,SalePrice,distance
1895,1896,1652,3,1,1Story,200000,0.007913
1940,1941,1647,3,1,1Story,153000,0.017804
618,619,1644,3,1,1Story,167000,0.023738
2700,2701,1640,3,1,1Story,131000,0.031651
2294,2295,1676,3,1,1Story,196000,0.039564
314,315,1687,3,1,1Story,160000,0.061324
788,789,1689,3,1,1Story,127500,0.065281
2282,2283,1622,3,1,1Story,168000,0.067259
1823,1824,1696,3,1,1Story,143900,0.079128
2298,2299,1608,3,1,1Story,80000,0.094954


**The results make sense because the closest homes have similar living area, bedrooms, bathrooms, and the same House Style as House 0. Adding House Style makes the recommendations more specific because homes with a different style are treated as less similar.
**

3\. Continuing parts 1 and 2. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it, by calculating distances. You can **choose the variables to include, but include both quantitative and categorical variables**. Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

_Hint:_ There are many variables in the data set. Do not attempt to compute distance based on all the variables! You will want to pare down the number of variables, but be sure to include a mixture of categorical and quantitative variables. Refer to the [data documentation](https://ww2.amstat.org/publications/jse/v19n3/decock/DataDocumentation.txt) for information about the variables.


In [4]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
numeric_vars = ["Gr Liv Area", "Full Bath", "Overall Qual", "Year Built"]
categorical_vars = ["House Style", "Neighborhood", "Bldg Type"]

house_0 = ames.iloc[0]
cheaper = ames[ames["SalePrice"] < house_0["SalePrice"]].copy()

numeric_data = ames[numeric_vars]
numeric_scaled = (numeric_data - numeric_data.mean()) / numeric_data.std()

categorical_data = pd.get_dummies(ames[categorical_vars], dtype=int)

profile_data = pd.concat([numeric_scaled, categorical_data], axis=1)

cheaper["euclidean_distance"] = (
    profile_data.loc[cheaper.index]
    .sub(profile_data.loc[0])
    .pow(2)
    .sum(axis=1)
    .pow(0.5)
)

cheaper["manhattan_distance"] = (
    profile_data.loc[cheaper.index]
    .sub(profile_data.loc[0])
    .abs()
    .sum(axis=1)
)

similar_homes = cheaper.sort_values("euclidean_distance").head(10)

display(
    similar_homes[
        ["Order", "Gr Liv Area", "Full Bath", "Overall Qual", "Year Built",
         "House Style", "Neighborhood", "Bldg Type", "SalePrice",
         "euclidean_distance", "manhattan_distance"]
    ]
)

,Order,Gr Liv Area,Full Bath,Overall Qual,Year Built,House Style,Neighborhood,Bldg Type,SalePrice,euclidean_distance,manhattan_distance
1895,1896,1652,1,6,1959,1Story,NAmes,1Fam,200000,0.033997,0.040976
147,148,1580,1,6,1959,1Story,NAmes,1Fam,159500,0.153936,0.183406
1924,1925,1721,1,6,1957,1Story,NAmes,1Fam,174850,0.162395,0.227772
1240,1241,1570,1,6,1958,1Story,NAmes,1Fam,166800,0.182525,0.236251
618,619,1644,1,6,1953,1Story,NAmes,1Fam,167000,0.232655,0.255179
617,618,1691,1,6,1967,1Story,NAmes,1Fam,175000,0.241575,0.300678
1231,1232,1537,1,6,1962,1Story,NAmes,1Fam,174000,0.244517,0.301532
605,606,1516,1,6,1964,1Story,NAmes,1Fam,167000,0.306906,0.409200
2585,2586,1575,1,6,1951,1Story,NAmes,1Fam,155000,0.337966,0.457801
2546,2547,1488,1,6,1964,1Story,NAmes,1Fam,167000,0.357686,0.464590


**I used living area, full bathrooms, overall quality, year built, house style, neighborhood, and building type to identify similar cheaper homes. The recommendations generally make sense because they match House 0 on several important physical and location characteristics. The results can change somewhat when using Manhattan distance instead of Euclidean distance, but the closest homes should still have similar overall profiles. Including both quantitative and categorical variables gives more realistic recommendations than using size and room counts by themselves.
**

## Colleges similar to Cal Poly

We'll use data from the [College Scorecard data](https://collegescorecard.ed.gov/) to find colleges and universities that are similar to Cal Poly.

In [5]:
df_college = pd.read_csv("https://datasci112.stanford.edu/data/college_attributes.csv")

df_college.set_index("Institution", inplace = True)

df_college

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,PCIP01,PCIP03,PCIP04,PCIP05,...,PCIP44,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,Normal,AL,0.7160,5098.0,Master's Colleges & Universities: Larger Programs,Public,0.0445,0.0071,0.0053,0.0000,...,0.0409,0.0249,0.0,0.0,0.0,0.0,0.0231,0.0000,0.1637,0.0000
University of Alabama at Birmingham,Birmingham,AL,0.8854,13284.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0020,...,0.0195,0.0239,0.0,0.0,0.0,0.0,0.0249,0.2088,0.2159,0.0141
University of Alabama in Huntsville,Huntsville,AL,0.7367,7358.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0127,0.0,0.0,0.0,0.0,0.0407,0.1341,0.1930,0.0073
Alabama State University,Montgomery,AL,0.9799,3495.0,Doctoral/Professional Universities,Public,0.0000,0.0000,0.0000,0.0000,...,0.0648,0.0196,0.0,0.0,0.0,0.0,0.0511,0.0904,0.1513,0.0059
The University of Alabama,Tuscaloosa,AL,0.7890,30725.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0061,0.0000,0.0019,...,0.0072,0.0661,0.0,0.0,0.0,0.0,0.0234,0.1077,0.2916,0.0096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Florida Academy of Nursing,Miramar,FL,0.3088,239.0,Not applicable,Private for-profit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,1.0000,0.0000,0.0000
Herzing University-Tampa,Tampa,FL,0.9630,68.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000
Abilene Christian University-Undergraduate Online,Addison,TX,1.0000,415.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000


We'll want to single out Cal Poly, which we can do like this.

In [6]:
school_name = "California Polytechnic State University-San Luis Obispo"

cp = df_college.loc[school_name]

cp

,California Polytechnic State University-San Luis Obispo
City,San Luis Obispo
State,CA
AdmissionRate,0.33
Undergraduates,21090.0
CarnegieClassification,Master's Colleges & Universities: Larger Programs
Ownership,Public
PCIP01,0.1084
PCIP03,0.0255
PCIP04,0.0441
PCIP05,0.0019


1\. Based on only the admission rate and the number of undergraduates, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [7]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
variables = ["AdmissionRate", "Undergraduates"]

college_data = df_college[variables].dropna()

scaled_data = (
    (college_data - college_data.mean()) / college_data.std()
)

distances = (
    scaled_data.sub(scaled_data.loc[school_name])
    .pow(2)
    .sum(axis=1)
    .pow(0.5)
)

closest_schools = distances.drop(school_name).sort_values().head(5)

display(
    df_college.loc[closest_schools.index, variables]
    .assign(distance=closest_schools)
)

,AdmissionRate,Undergraduates,distance
Institution,,,
University of California-Santa Barbara,0.2918,23081.0,0.309162
DeVry University-Illinois,0.4552,19729.0,0.593121
University of North Carolina at Chapel Hill,0.2040,19722.0,0.596846
Clemson University,0.4922,21577.0,0.736788
University of Virginia-Main Campus,0.2074,17041.0,0.761296


**I standardized admission rate and undergraduate enrollment, then used Euclidean distance to measure similarity to Cal Poly. The most similar school was UC Santa Barbara, followed by DeVry University–Illinois, UNC Chapel Hill, Clemson, and the University of Virginia. These schools are similar only on admission rate and enrollment; they may differ a lot in academic programs, location, and public/private status.
**

2\. Now consider the admission rate, the number of undergraduates, and also the [Carnegie classification](https://en.wikipedia.org/wiki/Carnegie_Classification_of_Institutions_of_Higher_Education) of the type of school, and the ownership (public, private, etc.) Based on these variables, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [8]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
numeric_vars = ["AdmissionRate", "Undergraduates"]
categorical_vars = ["CarnegieClassification", "Ownership"]

college_data = df_college[numeric_vars + categorical_vars].dropna()

numeric_scaled = (
    (college_data[numeric_vars] - college_data[numeric_vars].mean())
    / college_data[numeric_vars].std()
)

categorical_dummies = pd.get_dummies(
    college_data[categorical_vars], dtype=int
)

profiles = pd.concat([numeric_scaled, categorical_dummies], axis=1)

distances = (
    profiles.sub(profiles.loc[school_name])
    .pow(2)
    .sum(axis=1)
    .pow(0.5)
)

closest_schools = distances.drop(school_name).sort_values().head(5)

display(
    df_college.loc[
        closest_schools.index,
        numeric_vars + categorical_vars
    ].assign(distance=closest_schools)
)

,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,distance
Institution,,,,,
CUNY Hunter College,0.4590,17293.0,Master's Colleges & Universities: Larger Programs,Public,0.761441
CUNY Bernard M Baruch College,0.5056,15483.0,Master's Colleges & Universities: Larger Programs,Public,1.073601
CUNY John Jay College of Criminal Justice,0.4458,12834.0,Master's Colleges & Universities: Larger Programs,Public,1.184989
CUNY Brooklyn College,0.5136,12567.0,Master's Colleges & Universities: Larger Programs,Public,1.376321
University of California-Santa Barbara,0.2918,23081.0,Doctoral Universities: Very High Research Acti...,Public,1.447612


**I standardized admission rate and undergraduate enrollment, converted Carnegie classification and ownership into indicator variables, and used Euclidean distance. The most similar schools were CUNY Hunter College, CUNY Baruch College, CUNY John Jay College, CUNY Brooklyn College, and UC Santa Barbara. Adding the categorical variables favors schools that are public and have the same Carnegie classification as Cal Poly.
**

3\. The columns whose names begin with "PCIP" contain the proportions of students at each school studying various fields (e.g., Engineering, Psychology). Each field is represented by a two-digit code called the [CIP code](https://nces.ed.gov/ipeds/cipcode/browse.aspx?y=55).

If we only consider the proportions of students studying various fields, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [9]:
# YOUR CODE HERE. ADD CELLS AS NEEDED

pcip_vars = df_college.filter(regex="^PCIP").columns

pcip_data = df_college[pcip_vars].dropna()

distances = (
    pcip_data.sub(pcip_data.loc[school_name])
    .pow(2)
    .sum(axis=1)
    .pow(0.5)
)

closest_schools = distances.drop(school_name).sort_values().head(5)

display(
    df_college.loc[closest_schools.index, ["City", "State"]]
    .assign(distance=closest_schools)
)

,City,State,distance
Institution,,,
North Carolina State University at Raleigh,Raleigh,NC,0.084343
Iowa State University,Ames,IA,0.085054
University of Illinois Urbana-Champaign,Champaign,IL,0.111475
Mississippi State University,Mississippi State,MS,0.118799
Texas A & M University-College Station,College Station,TX,0.123973


**I used Euclidean distance across all PCIP proportions. Since each PCIP variable is already a proportion, no additional scaling was needed. The schools most similar to Cal Poly were North Carolina State University, Iowa State University, the University of Illinois Urbana-Champaign, Mississippi State University, and Texas A&M University.
**